# 美国上市餐饮企业历史市销率（P/S）

使用 Yahoo Finance（`yfinance`）下载尽可能长的日线价格、历史股份数和年度/季度营收，估算各公司的历史市销率。核心公司包括 CAKE、TXRH、DRI、SG 和 EAT；另加入全服务餐饮同业、快休闲邻近公司，以及 RUTH、CHUY 两家已退市公司的历史数据（如果 Yahoo 仍提供）。

## 口径与限制
- **P/S = 当日收盘价 × 当日可用历史流通股数 ÷ 最近已知 TTM 营收**。优先使用四个连续季度的营收合计；年度财报作为更早历史的补充。
- Yahoo Finance 的财报历史通常显著短于股价历史；此 notebook 不会把当前股份数冒充为历史股份数。若某股票缺少历史股份数或营收，就跳过无法计算的日期并记录原因。
- 财报可知日期按财报期末后 **60 天（季度）/120 天（年度）** 估算，避免直接在财报期末使用尚未公开的数据；这不是精确发布日期，仍可能有时间偏差。
- 股份数由 Yahoo 的历史序列按日向前填充；Yahoo 的拆股调整/历史股份口径可能与原始收盘价不完全一致。P/S 是估算值，适用于比较和研究，不应视为精确的历史市值数据。
- 不包含私营餐饮公司。Shake Shack、Wingstop、CAVA、Sweetgreen 是快休闲邻近同业，和传统全服务餐厅并非完全可比。RUTH 与 CHUY 作为退市历史样本单独标注。

数据来自 Yahoo Finance；网络请求失败、覆盖范围与财报口径差异会影响结果。请先运行全部单元格。

In [1]:
from __future__ import annotations

import time
import warnings
from datetime import date, timedelta
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import yfinance as yf

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")
plt.style.use("seaborn-v0_8-whitegrid")

# Yahoo Finance 请求间隔；遇到限流时可调大后重新运行。
REQUEST_DELAY_SECONDS = 0.35
QUARTERLY_REPORT_LAG_DAYS = 60
ANNUAL_REPORT_LAG_DAYS = 120
PS_DISPLAY_MAX = 30  # 仅限制图表显示范围，不截断导出的原始数据。
OUTPUT_CSV = Path.cwd() / "restaurant_historical_ps.csv"

COMPANIES = {
    "CAKE": ("The Cheesecake Factory", "核心：全服务休闲餐饮"),
    "TXRH": ("Texas Roadhouse", "核心：牛排馆/全服务休闲餐饮"),
    "DRI": ("Darden Restaurants", "核心：全服务及高端餐饮"),
    "SG": ("Sweetgreen", "核心：快休闲邻近同业"),
    "EAT": ("Brinker International", "核心：全服务休闲餐饮"),
    "BLMN": ("Bloomin' Brands", "扩展：全服务餐饮"),
    "CBRL": ("Cracker Barrel Old Country Store", "扩展：全服务餐饮"),
    "DIN": ("Dine Brands Global", "扩展：全服务餐饮/特许经营"),
    "PLAY": ("Dave & Buster's", "扩展：餐饮及娱乐"),
    "BJRI": ("BJ's Restaurants", "扩展：全服务餐饮"),
    "FWRG": ("First Watch Restaurant Group", "扩展：全服务餐饮"),
    "RRGB": ("Red Robin Gourmet Burgers", "扩展：全服务餐饮"),
    "LOCO": ("El Pollo Loco", "扩展：全服务/快速休闲餐饮"),
    "NATH": ("Nathan's Famous", "扩展：餐饮/特许经营"),
    "SHAK": ("Shake Shack", "邻近：快休闲餐饮"),
    "WING": ("Wingstop", "邻近：快休闲/特许经营"),
    "CAVA": ("CAVA Group", "邻近：快休闲餐饮"),
    "RUTH": ("Ruth's Hospitality Group", "历史退市：高端牛排馆"),
    "CHUY": ("Chuy's Holdings", "历史退市：全服务休闲餐饮"),
}
CORE_TICKERS = ["CAKE", "TXRH", "DRI", "SG", "EAT"]
HISTORICAL_DELISTED = ["RUTH", "CHUY"]

print(f"将尝试获取 {len(COMPANIES)} 家公司的 Yahoo Finance 数据。")
print(f"CSV 导出位置：{OUTPUT_CSV.resolve()}")

将尝试获取 19 家公司的 Yahoo Finance 数据。
CSV 导出位置：C:\Users\huada\stock_analyzer\yahoo_finance_agents\restaurant_historical_ps.csv


## 获取并计算

财报按报告可知日期排序后向前填充到交易日。年度收入仅在估计的披露滞后之后生效；季度 TTM 在有四个连续季度时优先使用。个股失败会记录在 `errors`，不影响其他股票继续处理。

In [ ]:
def _naive_datetime_index(index) -> pd.DatetimeIndex:
    """Normalize Yahoo timestamps to timezone-naive dates for alignment."""
    result = pd.DatetimeIndex(pd.to_datetime(index))
    if result.tz is not None:
        result = result.tz_localize(None)
    return result.normalize()


def _revenue_row(statement: pd.DataFrame) -> pd.Series:
    """Return the best available total/operating revenue row."""
    if statement is None or statement.empty:
        return pd.Series(dtype="float64")
    rows = {str(label).strip().lower(): label for label in statement.index}
    for label in ("total revenue", "operating revenue"):
        if label in rows:
            values = pd.to_numeric(statement.loc[rows[label]], errors="coerce")
            values.index = _naive_datetime_index(values.index)
            return values.dropna().sort_index()
    return pd.Series(dtype="float64")


def _revenue_observations(annual: pd.DataFrame, quarterly: pd.DataFrame) -> pd.DataFrame:
    """Build dated annual and quarterly-TTM revenue observations."""
    observations = []

    annual_revenue = _revenue_row(annual)
    for period_end, revenue in annual_revenue.items():
        observations.append({
            "known_date": period_end + pd.Timedelta(days=ANNUAL_REPORT_LAG_DAYS),
            "revenue": float(revenue),
            "source": "Annual revenue (estimated filing lag)",
        })

    quarterly_revenue = _revenue_row(quarterly)
    quarterly_revenue = quarterly_revenue[~quarterly_revenue.index.duplicated(keep="last")].sort_index()
    for position in range(3, len(quarterly_revenue)):
        window = quarterly_revenue.iloc[position - 3:position + 1]
        gaps = window.index.to_series().diff().dropna().dt.days
        if len(gaps) != 3 or not gaps.between(75, 110).all():
            continue
        observations.append({
            "known_date": window.index[-1] + pd.Timedelta(days=QUARTERLY_REPORT_LAG_DAYS),
            "revenue": float(window.sum()),
            "source": "Quarterly TTM (estimated filing lag)",
        })

    if not observations:
        return pd.DataFrame(columns=["known_date", "revenue", "source"])
    result = pd.DataFrame(observations).sort_values("known_date")
    result = result.drop_duplicates("known_date", keep="last")
    result = result[result["known_date"] <= pd.Timestamp(date.today())]
    return result[result["revenue"].gt(0)].reset_index(drop=True)


def fetch_company_ps(ticker: str, company: str, category: str) -> tuple[pd.DataFrame, dict]:
    """Fetch one company's price, historical shares and available revenue data."""
    instrument = yf.Ticker(ticker)
    prices = instrument.history(period="max", auto_adjust=False, actions=False, repair=True)
    if prices.empty or "Close" not in prices:
        raise ValueError("Yahoo Finance returned no daily Close price history")
    prices = prices.loc[pd.to_numeric(prices["Close"], errors="coerce").gt(0)].copy()
    prices.index = _naive_datetime_index(prices.index)
    prices = prices[~prices.index.duplicated(keep="last")].sort_index()

    start_date = prices.index.min().date().isoformat()
    end_date = (date.today() + timedelta(days=1)).isoformat()
    shares = instrument.get_shares_full(start=start_date, end=end_date)
    if shares is None or len(shares) == 0:
        raise ValueError("Yahoo Finance returned no historical shares; current shares will not be substituted")
    shares = pd.Series(shares, copy=True)
    shares.index = _naive_datetime_index(shares.index)
    shares = pd.to_numeric(shares, errors="coerce")
    shares = shares[shares.gt(0)].groupby(level=0).last().sort_index()
    shares_daily = shares.reindex(prices.index).ffill()

    source_warnings = []
    try:
        annual = instrument.get_income_stmt(freq="yearly", pretty=False)
    except Exception as exc:
        annual = pd.DataFrame()
        source_warnings.append(f"Annual income statement unavailable: {type(exc).__name__}: {exc}")
    try:
        quarterly = instrument.get_income_stmt(freq="quarterly", pretty=False)
    except Exception as exc:
        quarterly = pd.DataFrame()
        source_warnings.append(f"Quarterly income statement unavailable: {type(exc).__name__}: {exc}")
    revenue = _revenue_observations(annual, quarterly)
    if revenue.empty:
        raise ValueError("No positive annual or four-quarter TTM revenue observations were available")

    revenue_by_date = revenue.set_index("known_date").sort_index()
    revenue_daily = revenue_by_date["revenue"].reindex(prices.index, method="ffill")
    revenue_source_daily = revenue_by_date["source"].reindex(prices.index, method="ffill")
    market_cap = prices["Close"] * shares_daily
    ps = market_cap.div(revenue_daily).where(revenue_daily.gt(0))
    result = pd.DataFrame({
        "date": prices.index,
        "ticker": ticker,
        "company": company,
        "category": category,
        "close": prices["Close"].to_numpy(),
        "shares": shares_daily.to_numpy(),
        "market_cap": market_cap.to_numpy(),
        "ttm_revenue": revenue_daily.to_numpy(),
        "revenue_source": revenue_source_daily.to_numpy(),
        "ps_ratio": ps.to_numpy(),
    }).dropna(subset=["ps_ratio"])
    result = result[result["ps_ratio"].gt(0)].reset_index(drop=True)
    if result.empty:
        raise ValueError("No overlapping dates with valid price, historical shares and reported revenue")

    metadata = {
        "ticker": ticker,
        "company": company,
        "category": category,
        "price_start": prices.index.min().date(),
        "price_end": prices.index.max().date(),
        "ps_start": result["date"].min().date(),
        "ps_end": result["date"].max().date(),
        "ps_days": len(result),
        "annual_revenue_periods": len(_revenue_row(annual)),
        "quarterly_revenue_periods": len(_revenue_row(quarterly)),
        "source_warnings": source_warnings,
    }
    return result, metadata


all_results = []
coverage_rows = []
error_rows = []

for ticker, (company, category) in COMPANIES.items():
    try:
        frame, coverage = fetch_company_ps(ticker, company, category)
        all_results.append(frame)
        coverage_rows.append(coverage)
        print(f"{ticker}: {coverage['ps_start']} 至 {coverage['ps_end']}，{coverage['ps_days']:,} 个交易日")
        for warning in coverage["source_warnings"]:
            print(f"  财报数据警告：{warning}")
    except Exception as exc:
        error_rows.append({"ticker": ticker, "company": company, "error": f"{type(exc).__name__}: {exc}"})
        print(f"{ticker}: 获取失败 - {type(exc).__name__}: {exc}")
    time.sleep(REQUEST_DELAY_SECONDS)

if not all_results:
    raise RuntimeError("没有公司成功获取可计算的历史 P/S；请检查网络、yfinance 版本或 Yahoo 数据可用性。")

ps_history = pd.concat(all_results, ignore_index=True).sort_values(["ticker", "date"])
coverage = pd.DataFrame(coverage_rows).sort_values("ticker").reset_index(drop=True)
errors = pd.DataFrame(error_rows, columns=["ticker", "company", "error"])

print(f"\n成功：{len(coverage)} 家；失败：{len(errors)} 家；P/S 记录：{len(ps_history):,} 行")

CAKE: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available
TXRH: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available
DRI: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available
SG: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available
EAT: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available
BLMN: 获取失败 - ValueError: No positive annual or four-quarter TTM revenue observations were available


## 覆盖范围和数据质量

下面的覆盖摘要用于确认 Yahoo 实际提供的数据区间。财报数量较少是 Yahoo 数据覆盖的限制，不代表公司历史较短。

In [ ]:
display(coverage)

if not errors.empty:
    print("\n失败股票及原因：")
    display(errors)
else:
    print("所有配置的股票均成功获取可计算数据。")

print("\n每家公司的 P/S 历史覆盖：")
display(
    ps_history.groupby(["ticker", "company", "category"], as_index=False)
    .agg(first_date=("date", "min"), last_date=("date", "max"), observations=("ps_ratio", "size"))
    .sort_values(["category", "ticker"])
)

## 绘制历史 P/S

用月末最后一个可用交易日的 P/S 绘图，以便比较长期趋势；CSV 仍保留全部日频观测。图表上限仅为可读性设置，超过上限的观测保留在数据和 CSV 中。

In [ ]:
monthly_ps = (
    ps_history.set_index("date")
    .groupby("ticker")["ps_ratio"]
    .resample("ME")
    .last()
    .dropna()
    .unstack("ticker")
)

plot_groups = {
    "核心公司：全服务与快休闲餐饮": CORE_TICKERS,
    "扩展同业：全服务餐饮": [ticker for ticker, (_, group) in COMPANIES.items() if group.startswith("扩展：")],
    "邻近同业：快休闲餐饮": [ticker for ticker, (_, group) in COMPANIES.items() if group.startswith("邻近：")],
    "历史退市公司": HISTORICAL_DELISTED,
}

for title, tickers in plot_groups.items():
    available = [ticker for ticker in tickers if ticker in monthly_ps.columns]
    if not available:
        print(f"{title}：无可绘制数据")
        continue
    ax = monthly_ps[available].plot(figsize=(14, 6), linewidth=1.5)
    ax.set_title(title)
    ax.set_xlabel("日期")
    ax.set_ylabel("P/S（历史市值 / TTM 营收）")
    ax.set_ylim(bottom=0, top=PS_DISPLAY_MAX)
    ax.legend(title="Ticker", ncol=2)
    plt.tight_layout()
    plt.show()

## 最新可用估值与导出

“最新”指该股票最近一个同时具备有效股价、历史股份数和可用 TTM/年度营收的交易日；不同股票的数据日期可能不同。

In [ ]:
latest_ps = (
    ps_history.sort_values("date")
    .groupby("ticker", as_index=False)
    .tail(1)
    [["ticker", "company", "category", "date", "close", "shares", "market_cap", "ttm_revenue", "revenue_source", "ps_ratio"]]
    .sort_values("ps_ratio", ascending=False)
    .reset_index(drop=True)
)
display(latest_ps)

ps_history.to_csv(OUTPUT_CSV, index=False, date_format="%Y-%m-%d", encoding="utf-8-sig")
print(f"\n已导出 {len(ps_history):,} 条日频 P/S 记录：{OUTPUT_CSV.resolve()}")